In [ ]:
import numpy as np
import sys
import sys; sys.path.append('..');sys.path.append('../periodic_patches');sys.path.append('../periodic_patches/Visualization/');sys.path.append('../gmsh');sys.path.append('../periodic_patches/experiments/parametrization_experiments/');

In [ ]:
import inflation, sparse_matrices, mesh, numpy as np, importlib, pickle
import inflatables_parametrization as parametrization
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization

In [ ]:
import visualize_stiffness, importlib
importlib.reload(visualize_stiffness)

In [ ]:
import parametrization_experiment_helper, importlib
importlib.reload(parametrization_experiment_helper)

In [ ]:
pattern = parametrization_experiment_helper.Pattern_data[1]

In [ ]:
experiment_file = pattern['experiment_file']
stiffness_path = pattern['stiffness_path']
pattern_name = pattern['name']
num_pattern_params = pattern['num_pattern_params']
param_index = pattern['param_index']
default_param = pattern['default_param']
param_range = pattern['param_range']
param_normalization_factor = pattern['param_normalization_factor']
fusing_curve_polyline = pattern['fusing_curve_polyline_function']
                                

In [ ]:
shape = parametrization_experiment_helper.Shape_data[3]

In [ ]:
shape_name = shape['name']
shape_path = shape['path']

### Overview

In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np
import visualize_stiffness

In [ ]:
with open(experiment_file, 'r') as fp:
    data = json.load(fp)

In [ ]:
df = pd.DataFrame(data['data'])
valid_tags = np.array(df['name'][df['Planar equilibrium'] == 1])

In [ ]:
kappa_path = None

In [ ]:
bending_stiffness_data, stretching_stiffness_data, scale_factor_data, used_tags = visualize_stiffness.plot_all_data(kappa_path, stiffness_path, pattern_name, valid_tags, plot_data = False)

In [ ]:
parameters = []
for index in param_index:
    parameters.append(data['pattern_parameters'][index]['values'])

In [ ]:
max_bending_stiffness = np.max(bending_stiffness_data, axis = 1)
min_bending_stiffness = np.min(bending_stiffness_data, axis = 1)
max_stretching_stiffness = np.max(stretching_stiffness_data, axis = 1)
min_stretching_stiffness = np.min(stretching_stiffness_data, axis = 1)

In [ ]:
x_scale_factors, y_scale_factors = visualize_stiffness.get_axis_scale_factors(stiffness_path, pattern_name, valid_tags)

In [ ]:
min_scale_factors = np.min(np.concatenate((x_scale_factors.reshape(-1, 1), y_scale_factors.reshape(-1, 1)), axis = 1), axis = 1)
max_scale_factors = np.max(np.concatenate((x_scale_factors.reshape(-1, 1), y_scale_factors.reshape(-1, 1)), axis = 1), axis = 1)

In [ ]:
angle_offsets = visualize_stiffness.get_max_flattening_factor_offset(stiffness_path, pattern_name, valid_tags)

### Get scale function convex hull

In [ ]:
import matplotlib.cm as cm
import matplotlib as mpl

In [ ]:
from scipy.spatial import ConvexHull, convex_hull_plot_2d
import numpy as np
rng = np.random.default_rng()
points = rng.random((30, 2))   # 30 random points in 2-D
# points = np.concatenate((min_scale_factor.reshape((-1, 1)), max_scale_factor.reshape((-1, 1))), axis = 1)

points = np.concatenate((max_scale_factors.reshape((-1, 1)), min_scale_factors.reshape((-1, 1))), axis = 1)
hull = ConvexHull(points)

### Validate the max and min scale factors are aligned with the x and y axis

In [ ]:
import parametrization_helper

In [ ]:
eqns = hull.equations

In [ ]:
parametrization_helper.visualize_scale_factors(eqns, max_scale_factors, min_scale_factors)

### Generate data without augmenting

In [ ]:
stiffness_coefficients = np.array(visualize_stiffness.get_stiffness_coefficients(stiffness_path, pattern_name, (used_tags)))
# For patches with reflection symmetry:
stiffness_coefficients[:, 1] *= 0
stiffness_coefficients[:, 2] *= 0

In [ ]:
grid_shape = np.zeros(len(parameters) + 1, dtype = np.int64)
grid_shape[0] = 9
for i in range(len(parameters)):
    grid_shape[i + 1] = len(parameters[i])

In [ ]:
grid_shape

In [ ]:
grid_data = np.zeros(tuple(grid_shape))

In [ ]:
stiffness_coefficients.reshape([*grid_shape[1:], 5]).shape

In [ ]:
# np.transpose(grid_stiffness_coefficients, (len(grid_stiffness_coefficients.shape)-1,) + tuple(range(len(grid_stiffness_coefficients.shape)-1)))[0]

In [ ]:
grid_stiffness_coefficients = stiffness_coefficients.reshape([*grid_shape[1:], 5])
grid_stiffness_coefficients = np.transpose(grid_stiffness_coefficients, (len(grid_stiffness_coefficients.shape)-1,) + tuple(range(len(grid_stiffness_coefficients.shape)-1)))


In [ ]:
grid_data[0] = max_scale_factors.reshape(grid_shape[1:])
grid_data[1] = min_scale_factors.reshape(grid_shape[1:])
grid_data[2] = x_scale_factors.reshape(grid_shape[1:])
grid_data[3] = y_scale_factors.reshape(grid_shape[1:])
grid_stiffness_coefficients = stiffness_coefficients.reshape([*grid_shape[1:], 5])
grid_stiffness_coefficients = np.transpose(grid_stiffness_coefficients, (len(grid_stiffness_coefficients.shape)-1,) + tuple(range(len(grid_stiffness_coefficients.shape)-1)))
for s in range(5):
    grid_data[4 + s] = grid_stiffness_coefficients[s]

In [ ]:
importlib.reload(parametrization_helper)

In [ ]:
grid_data.shape

In [ ]:
if len(parameters) == 1:
    splines = parametrization_helper.scipy_get_mat_params_over_one_pattern_params_grid_interpolation(parameters, grid_data)
elif len(parameters) == 2:
    splines = parametrization_helper.scipy_get_mat_params_over_pattern_params_grid_interpolation(parameters[0], parameters[1], grid_data)
else:
    splines = parametrization_helper.ndsplines_get_mat_params_over_pattern_params_grid_interpolation(grid_data, *parameters)


In [ ]:
import sys; sys.path.append('..')
import MeshFEM
import inflatables_parametrization as parametrization, sparse_matrices, mesh, numpy as np
from numpy.linalg import norm
import visualization

In [ ]:
import MeshFEM, parallelism, benchmark, utils
parallelism.set_max_num_tbb_threads(1)
parallelism.set_gradient_assembly_num_threads(1)
parallelism.set_hessian_assembly_num_threads(1)

In [ ]:
m = mesh.Mesh("../../examples/igloo.obj")
# m = mesh.Mesh("../../examples/cone_test.obj")
lg = parametrization.LocalGlobalGenericParametrizer(m, parametrization.lscm(m))
# lg = parametrization.LocalGlobalParametrizer(m, parametrization.lscm(m))


In [ ]:
# import tri_mesh_viewer
# view = tri_mesh_viewer.TriMeshViewer(m)
# view.show()

In [ ]:
for i in range(1000):
    lg.runIteration()
lg.energy()

In [ ]:
lg.alphaMin = 1.1
lg.alphaMax = 2.0

lg.betaMin = 1.1
lg.betaMax = 2.0
lg.energy()

In [ ]:
lg.runIteration()
lg.energy()

In [ ]:
rgp = parametrization.RegularizedGenericParametrizer(lg)

In [ ]:
rgp.stretchRegW = 0

In [ ]:
rgp.useBarrier

In [ ]:
import fd_validation

In [ ]:
fd_validation.gradConvergencePlot(rgp, epsilons = np.logspace(-9, -1, 100))

In [ ]:
# fd_validation.hessConvergencePlot(rgp, epsilons = np.logspace(-9, -1, 100))

### Pattern parameters optimization

In [ ]:
default_pattern_params = []
for i in range(len(parameters)):
    default_pattern_params += [default_param[i]]  * len(lg.getAlphas())

In [ ]:
rparam = parametrization.RegularizedPatternParametrizer(lg, splines, default_pattern_params, len(grid_data.shape) - 1)
rparam.patternParamBounds = np.array(param_range)
rparam.diffRegW = 0.0

In [ ]:
rparam.hessian()

In [ ]:
# perturb = np.random.uniform(low=-1,high=1, size=rsvd.numVars())
# fixedVars = np.arange(rsvd.phiOffset(), rsvd.numVars())
# # fixedVars = []

In [ ]:
import fd_validation
# fd_validation.validateGrad(rsvd, fd_eps=1e-8,
#                            xeval=rsvd.getVars() + 1e-3 * np.random.uniform(low=-1, high=1, size=rsvd.numVars()),
#                            perturb=perturb[0:rsvd.numVars()], fixedVars=fixedVars)

In [ ]:
rparam.energy(energyType = rparam.PatternEnergyType.Full)

In [ ]:
rparam.gradient(energyType = rparam.PatternEnergyType.Full)

In [ ]:
rparam.setVars(rparam.getVars())

In [ ]:
perturb = np.random.uniform(low = -1, high = 1, size = rparam.numVars())

In [ ]:
perturb[rparam.stretchOffset():] *= 0
# perturb[:rsvd.rgp.stretchOffset()] *= 0

In [ ]:
perturb

In [ ]:
fd_validation.gradConvergencePlot(rparam, customArgs = {"energyType": rparam.PatternEnergyType.RGP}, perturb = perturb)

In [ ]:
fd_validation.gradConvergencePlot(rparam, customArgs = {"energyType": rparam.PatternEnergyType.RGP}, epsilons = np.logspace(-9, -1, 100))

In [ ]:
fd_validation.gradConvergencePlot(rparam, customArgs = {"energyType": rparam.PatternEnergyType.Bending}, epsilons = np.logspace(-9, -1, 100))

In [ ]:
fd_validation.gradConvergencePlot(rparam, customArgs = {"energyType": rparam.PatternEnergyType.PatternRegularization})

In [ ]:
fd_validation.gradConvergencePlot(rparam, customArgs = {"energyType": rparam.PatternEnergyType.Full})

In [ ]:
fd_validation.hessConvergencePlot(rparam, customArgs = {"energyType": rparam.PatternEnergyType.Bending})

In [ ]:
fd_validation.hessConvergencePlot(rparam, customArgs = {"energyType": rparam.PatternEnergyType.RGP})

In [ ]:
var_types = ['beforeStretching', 'p1', 'p2']
var_indices = {'beforeStretching': np.arange(rparam.stretchOffset()),
               'p1': np.arange(rparam.stretchOffset(), rparam.stretchOffset() + len(lg.getAlphas()), 1),
               'p2': np.arange(rparam.stretchOffset() + len(lg.getAlphas()), rparam.numVars(), 1)}

In [ ]:
# fd_validation.hessian_convergence_block_plot(rsvd, var_types, var_indices, customArgs = {"energyType": rsvd.PatternEnergyType.RGP})

In [ ]:
fd_validation.hessian_convergence_block_plot(rparam, var_types, var_indices, customArgs = {"energyType": rparam.PatternEnergyType.Bending})

In [ ]:
fd_validation.hessConvergencePlot(rparam, customArgs = {"energyType": rparam.PatternEnergyType.PatternRegularization})